> # ⚠️ Do not run this notebook unattended
>
> The last section, **`# Hardware validation`**, connects to the real board, uploads the
> package and reads the TMP102 on I2C bus 1 through the TCA9548A (read-only transfers, plus
> selecting and releasing mux channel 2). *Run All* and `jlx run` reach that section.
>
> - **Safe to run on its own:** everything above `# Hardware validation`. It runs on the host
>   against `micropython_i2c.sim.bus` parts, writes the module and its tests, and needs no
>   board.
> - **Hardware section:** run it one cell at a time, reading the **What to do** line above
>   each cell first.

**Where each cell runs.** Everything above `%board connect` runs in ordinary Python
(CPython, the project `.venv`) on this computer. After `%board connect`, cells run in
MicroPython on the board, except cells that start with `%%host`.

Notebook-only: import the package before anything else. Its `__init__` imports `protocol`,
and the header cell below rewrites `protocol.py`; importing the package first means no later
import reads the half-written file.

In [ ]:
import micropython_i2c  # noqa: F401

# Hardware and configuration

| | Host run (simulation) | Hardware run |
|---|---|---|
| **Runs on** | CPython 3.13 in the micropylab kernel, no board | Unexpected Maker FeatherS3 (ESP32-S3), MicroPython 1.28.0, USB `/dev/ttyACM0` |
| **I2C bus** | `FakeBus` | `machine.I2C(1)`, **bus 1**, scl=15 sda=16, 100 kHz (`bus_open("feathers3", 1)`) |
| **Mux** | none | SparkFun Qwiic Mux (TCA9548A) at `0x70`, channel 2 |
| **Register part** | `FakeRegisterPart` at `0x48` laid out like a TMP102 (16-bit registers at `0x00`, `0x02`, `0x04`; configuration `0x60A0` with read-only resolution and status bits) | TMP102 at `0x48`: temperature and configuration registers read, never written |
| **Command part** | `FakeSensirion` at `0x62`, words `0xBEEF`, `0x1234`, Sensirion CRC-8 | a stand-in bus on the board (no command part on bus 1) |
| **Result** | not run yet | not run yet |

# TODO:
Nothing open.

Decided (2026-11-16): the protocol mixins moved from a hand-written file into this notebook.
Their public API and behaviour are unchanged: every driver depends on them. The tests no
longer use the TMP102 driver and its fake: they run on a generic `FakeRegisterPart` laid out
like a TMP102, so this base layer depends on no device.

# Python file
location: `src/micropython_i2c/protocol.py`

tests: `tests/test_protocol.py`

## header

In [ ]:
%%extract ../src/micropython_i2c/protocol.py
"""Protocol mixins: ``RegisterI2CBase`` (register-mapped) and ``CommandI2CBase`` (opcode).

Both call only the four transport methods from ``I2CNode`` and never take a lock, so
they compose with ``I2CDevice`` *or* ``I2CMux``. Neither defines ``__init__``, which
keeps the MRO trivial for driver authors::

    class Sensor(RegisterI2CBase, I2CDevice): ...
    class GasSensor(CommandI2CBase, I2CDevice): ...
    class CheckedSensor(RegisterI2CBase, CRCMixin, I2CDevice): ...

Naming is ``noun_verb`` throughout (``reg_read``, ``field_set``, ``crc_verify``) so
related operations sort together.

Rule of thumb for where a fact lives: if the datasheet states it once for the chip it is
a class attribute (endianness, default register width, opcode width, CRC parameters);
if it varies per register it is a per-call argument (``nbytes``, ``tolerance``).

Generated by ``dev_notebooks/protocol_devNB``: edit the notebook, not this file.
"""

# Imports
## External
None.

## Internal
`BitTools` (byte packing, bitfields) under `RegisterI2CBase`; `CRCMixin` (which is a
`BitTools` too) under `CommandI2CBase`.

In [ ]:
%%extract ../src/micropython_i2c/protocol.py -a
from micropython_i2c.bits import BitTools
from micropython_i2c.crc import CRCMixin

# Setup

Notebook-only: run the test cells in the notebook. `raise_on_error=True` makes a failing test
fail the run.

In [ ]:
import ipytest

ipytest.autoconfig(raise_on_error=True)

# Test file
The test file header: imports only. It is written, not run.

In [ ]:
%%writefile ../tests/test_protocol.py
import pytest

from micropython_i2c.connection import I2CConnection
from micropython_i2c.node import I2CDevice
from micropython_i2c.protocol import CommandI2CBase, RegisterI2CBase
from micropython_i2c.sim.bus import FakeBus, FakeRegisterPart, FakeSensirion

Notebook-only: the imports the validation and test cells use.

In [ ]:
import pytest

from micropython_i2c.connection import I2CConnection
from micropython_i2c.node import I2CDevice
from micropython_i2c.sim.bus import FakeBus, FakeRegisterPart, FakeSensirion

# Intent
**Purpose:** the two ways a part is talked to, written once so drivers only declare their
register map or command set.

| Mixin | Parts | What it gives a driver |
|---|---|---|
| `RegisterI2CBase` | register-mapped (TMP102, TMAG5273, VL53L1X, ...) | raw and managed register access, a shadow cache, verified writes, bitfields, coded fields |
| `CommandI2CBase` | command-driven (SHTC3 and other Sensirion parts) | send an opcode, read 16-bit words each followed by a CRC |

**Where they sit.** Both are mixins over the transport contract of `node.I2CNode`
(`read_mem`, `write_mem`, `read_raw`, `write_raw`). They never touch a bus, an address or a
lock, and neither defines `__init__`:

```mermaid
classDiagram
    BitTools <|-- RegisterI2CBase
    BitTools <|-- CRCMixin
    CRCMixin <|-- CommandI2CBase
    RegisterI2CBase <|-- Sensor
    I2CDevice <|-- Sensor
    CommandI2CBase <|-- GasSensor
    I2CDevice <|-- GasSensor
```

**Used by:** every driver in `devices`.

## What the module defines

| Name | What it is |
|---|---|
| `RegisterI2CBase` | register-mapped protocol: `reg_read`/`reg_write` (raw), `reg_get`/`reg_set` (cached, verified), fields, flags, coded fields |
| `CommandI2CBase` | command protocol: `cmd_send`, `words_read` with a CRC per word |

# Code
## TODO:
Nothing

## Public names

In [ ]:
%%extract ../src/micropython_i2c/protocol.py -a
__all__ = ["CommandI2CBase", "RegisterI2CBase"]

## `RegisterI2CBase`
Two access pairs, and the difference matters:

| Pair | Cache | Verify | Use for |
|---|---|---|---|
| `reg_read` / `reg_write` | no | no | write-only registers, reads with side effects, write-1-to-clear bits, handshake bits that read back inverted |
| `reg_get` / `reg_set` | `reg_get` serves from it; `reg_set` stores the confirmed value | `reg_set` reads back and compares | configuration: the default |

**Verification** compares only the bits expected to read back as written. The mask is the
per-call `verify_mask`, else `VERIFY_MASKS[reg]`, else `VERIFY_MASK`, else every bit.
`tolerance` switches to a numeric comparison $|\text{read-back} - \text{value}| \le
\text{tolerance}$, for registers the part quantises.

**The cache** stores `(nbytes, value)` per register, so a read at another width is a miss,
not a stale-width hit. It is created on first use (no `__init__`). It is only as good as the
driver's discipline: cache configuration, and pass `refresh=True` for anything measured.

**Fields** are `(shift, width)` tuples on the driver class; `field_get` reads fresh by
default because fields sit next to live status bits. **Coded fields** map a bit pattern to a
physical value through a `{code: value}` table.

In [ ]:
%%extract ../src/micropython_i2c/protocol.py -a
class RegisterI2CBase(BitTools):
    """Register-mapped devices: shadow cache, confirmed writes, bitfields, coded fields.

    Two access pairs, and the difference matters:

    * ``reg_read`` / ``reg_write`` are **raw**: no cache, no verification. For
      write-only registers, reads with side effects, write-1-to-clear bits and
      handshake bits that read back inverted.
    * ``reg_get`` / ``reg_set`` are **managed**: ``reg_get`` serves from the shadow
      cache; ``reg_set`` reads back from the device, compares, raises on mismatch and
      caches only the confirmed value. These are the default.

    Verification compares only the bits expected to read back as written. The mask is
    the per-call ``verify_mask``, else ``VERIFY_MASKS[reg]``, else ``VERIFY_MASK``,
    else every bit. ``tolerance`` switches to a numeric comparison for registers the
    part quantises (DACs, current limits); it and a mask are mutually exclusive.

    The cache is only as trustworthy as the driver's discipline: cache configuration,
    and pass ``refresh=True`` for anything measured, status-like or self-clearing.

    Class attributes:
        BIG_ENDIAN: Byte order of multi-byte registers.
        REG_BYTES: Default register width; override per call with ``nbytes``.
        VERIFY_MASK: Chip-wide verify mask, or None for all bits.
        VERIFY_MASKS: ``{reg: mask}`` for registers that need their own mask.
    """

    BIG_ENDIAN = True
    REG_BYTES = 2
    VERIFY_MASK = None
    VERIFY_MASKS = {}

    # Cache: {reg: (nbytes, value)}. Width is stored so a read at a different width is
    # a miss rather than a stale-width hit. Created lazily so no __init__ is needed.

    @property
    def reg_cache(self):
        """The shadow cache, ``{reg: (nbytes, value)}``. Created on first access."""
        try:
            return self._rc
        except AttributeError:
            self._rc = {}
            return self._rc

    def cached(self, reg):
        """Return the cached value of ``reg``, or None if it is not cached."""
        entry = self.reg_cache.get(reg)
        return None if entry is None else entry[1]

    def cache_clear(self, reg=None):
        """Drop ``reg`` from the cache, or everything if ``reg`` is None.

        Use after anything that changes the device behind the driver's back: a
        general-call reset, a power cycle, a reset line.
        """
        if reg is None:
            self.reg_cache.clear()
        else:
            self.reg_cache.pop(reg, None)

    def _width(self, nbytes):
        return self.REG_BYTES if nbytes is None else nbytes

    # Raw access.

    def reg_read(self, reg, nbytes=None):
        """Read a register and assemble it into an int. Always hits the bus."""
        nbytes = self._width(nbytes)
        return self.bytes_to_int(self.read_mem(reg, nbytes), self.BIG_ENDIAN)

    def reg_write(self, reg, value, nbytes=None):
        """Write an int to a register (truncated to width). No verify, no cache update."""
        nbytes = self._width(nbytes)
        self.write_mem(reg, self.int_to_bytes(value, nbytes, self.BIG_ENDIAN))

    # Managed access.

    def reg_get(self, reg, nbytes=None, refresh=False):
        """Return a register's value, from the cache when it holds one at this width."""
        nbytes = self._width(nbytes)
        entry = self.reg_cache.get(reg)
        if refresh or entry is None or entry[0] != nbytes:
            entry = (nbytes, self.reg_read(reg, nbytes))
            self.reg_cache[reg] = entry
        return entry[1]

    def verify_mask_for(self, reg):
        """The mask ``reg_set`` uses for ``reg`` when none is passed (None = all bits)."""
        return self.VERIFY_MASKS.get(reg, self.VERIFY_MASK)

    def reg_set(self, reg, value, nbytes=None, tolerance=0, verify_mask=None, validator=None):
        """Write a register, read it back from the device, and confirm it took.

        Args:
            reg: Register number.
            value: Value requested.
            nbytes: Register width; defaults to ``REG_BYTES``.
            tolerance: Largest acceptable ``|read-back - value|``. 0 means bitwise.
            verify_mask: Compare only these bits. Defaults to ``verify_mask_for(reg)``.
            validator: Called as ``validator(value)`` before any bus traffic; should
                raise on a bad value, so a rejected request costs nothing.

        Returns:
            The value read back, which may differ from the request within tolerance
            or in masked-off bits.

        Raises:
            ValueError: If the read-back fails the comparison.
        """
        if validator is not None:
            validator(value)
        nbytes = self._width(nbytes)
        mask = verify_mask if verify_mask is not None else self.verify_mask_for(reg)

        self.reg_write(reg, value, nbytes)
        actual = self.reg_read(reg, nbytes)

        if tolerance:
            ok = abs(actual - value) <= tolerance
            detail = "tol {}".format(tolerance)
        elif mask is not None:
            ok = (actual & mask) == (value & mask)
            detail = "mask {:#x}".format(mask)
        else:
            ok = actual == value
            detail = "exact"
        if not ok:
            raise ValueError(
                "reg {:#04x}: wrote {:#x}, device holds {:#x} ({})".format(
                    reg, value, actual, detail
                )
            )
        self.reg_cache[reg] = (nbytes, actual)
        return actual

    # Bitfields: ``field`` is a (shift, width) tuple declared on the driver class.

    def field_get(self, reg, field, nbytes=None, refresh=True):
        """Read a field. Fresh by default, since fields sit beside live status bits."""
        return self.field_extract(self.reg_get(reg, nbytes, refresh=refresh), field)

    def field_set(self, reg, field, value, nbytes=None, **kw):
        """Read-modify-write one field from a fresh read, verified via ``reg_set``.

        Extra keyword arguments (notably ``verify_mask``) go to ``reg_set``.
        """
        current = self.reg_read(reg, nbytes)
        return self.reg_set(reg, self.field_insert(current, field, value), nbytes, **kw)

    def flag_get(self, reg, field, refresh=True):
        """Read a one-bit field as a bool."""
        return bool(self.field_get(reg, field, refresh=refresh))

    def flag_set(self, reg, field, enable, **kw):
        """Write a one-bit field from a truthy value."""
        return self.field_set(reg, field, 1 if enable else 0, **kw)

    # Coded fields: the bit pattern indexes a {code: physical value} table.

    def coded_get(self, reg, field, table, nbytes=None, refresh=True):
        """Read a coded field and map it through ``table``."""
        return table[self.field_get(reg, field, nbytes, refresh=refresh)]

    def coded_set(self, reg, field, table, value, nbytes=None, **kw):
        """Map ``value`` back to its code and write it.

        Raises:
            ValueError: Listing the accepted values, if ``value`` is not in ``table``.
        """
        for code, v in table.items():
            if v == value:
                return self.field_set(reg, field, code, nbytes, **kw)
        raise ValueError("{!r} not in {}".format(value, sorted(table.values())))

### validation/test
A demo register map for the cells below, shaped like a TMP102 so the tests check the same
things the TMP102-based tests did before (2026-11-16): 16-bit big-endian registers, a
configuration register whose resolution bits `RES` (bits 14..13) always read back `11`, a
status bit `OS` (15) and an alert bit `AL` (5) that a write does not set. `FakeRegisterPart`
is a file of 8-bit registers with auto-increment, so a 16-bit register at `n` occupies bytes
`n` and `n + 1`: the registers sit at every other address.

| Register | Address | Starts at | Read-only bits |
|---|---|---|---|
| `REG_TEMP` | `0x00` | `0x1900` (25 C in TMP102 counts) | none |
| `REG_CONFIG` | `0x02` | `0x60A0` | `OS`, `RES`, `AL` |
| `REG_THIGH` | `0x04` | `0x5000` | none |

In [ ]:
%%extract ../tests/test_protocol.py -a
class DemoRegs(RegisterI2CBase, I2CDevice):
    """A TMP102-shaped register map on a generic register part."""

    ALLOWED_ADDRS = (0x48,)
    REG_TEMP = 0x00
    REG_CONFIG = 0x02
    REG_THIGH = 0x04
    F_OS = (15, 1)  # status: reads 0 here, ignored by the verify mask
    F_RES = (13, 2)  # read-only, always 0b11
    F_SD = (8, 1)
    F_CR = (6, 2)
    F_AL = (5, 1)  # read-only
    CONV_RATES = {0: 0.25, 1: 1.0, 2: 4.0, 3: 8.0}
    CFG_POR = 0x60A0
    CONFIG_VERIFY_MASK = 0xFFFF & ~((1 << 15) | (3 << 13) | (1 << 5))
    VERIFY_MASKS = {REG_CONFIG: CONFIG_VERIFY_MASK}


def demo_part():
    """The register file behind ``DemoRegs``: TEMP 0x1900, CONFIG 0x60A0, THIGH 0x5000."""
    regs = {0x00: 0x19, 0x01: 0x00, 0x02: 0x60, 0x03: 0xA0, 0x04: 0x50, 0x05: 0x00}
    return FakeRegisterPart(regs, read_only={0x02: 0xE0, 0x03: 0x20})


@pytest.fixture
def proto():
    bus = FakeBus()
    bus.attach(0x48, demo_part())
    return DemoRegs(I2CConnection(bus, 0x48))

#### validation: nominal usage
Raw reads, the cache (miss, hit, refresh), a verified write that lands, a field and a coded
field.

In [ ]:
bus = FakeBus()
part = bus.attach(0x48, demo_part())
dev = DemoRegs(I2CConnection(bus, 0x48))
print("config: {:#06x} | temp: {:#06x}".format(dev.reg_read(dev.REG_CONFIG), dev.reg_read(dev.REG_TEMP)))
print("cached before:", dev.cached(dev.REG_TEMP), "| reg_get:", hex(dev.reg_get(dev.REG_TEMP)))
part.regs[0] = 0x1A  # the part converts again: 26 C
print("reg_get (stale by design): {:#06x} | refresh: {:#06x}".format(
    dev.reg_get(dev.REG_TEMP), dev.reg_get(dev.REG_TEMP, refresh=True)))  # fmt: skip
print("reg_set THIGH 0x1E00 ->", hex(dev.reg_set(dev.REG_THIGH, 0x1E00)), "| cache:", dev.reg_cache)
print("RES field:", dev.field_get(dev.REG_CONFIG, dev.F_RES), "| rate:",
      dev.coded_get(dev.REG_CONFIG, dev.F_CR, dev.CONV_RATES), "Hz")  # fmt: skip
dev.flag_set(dev.REG_CONFIG, dev.F_SD, True)
print("after shutdown flag: {:#06x}, SD = {}".format(
    dev.reg_read(dev.REG_CONFIG), dev.flag_get(dev.REG_CONFIG, dev.F_SD)))  # fmt: skip

#### validation: anti-nominal usage
A write that does not read back is refused, and the cache keeps the last confirmed value.
Here: clearing `RES`, compared on every bit.

In [ ]:
bus = FakeBus()
bus.attach(0x48, demo_part())
dev = DemoRegs(I2CConnection(bus, 0x48))
dev.reg_get(dev.REG_CONFIG)
try:
    dev.reg_set(dev.REG_CONFIG, 0x00A0, verify_mask=0xFFFF)  # RES is read-only, reads back 11
except ValueError as err:
    print("raised ValueError, as expected:\n    {}".format(err))
else:
    raise AssertionError("clearing RES with a full verify mask should have raised ValueError")
finally:
    print("cache still holds the confirmed value: {:#06x}".format(dev.cached(dev.REG_CONFIG)))

A value outside a coded field's table is refused before any bus traffic.

In [ ]:
n = len(bus.log)
try:
    dev.coded_set(dev.REG_CONFIG, dev.F_CR, dev.CONV_RATES, 2.0)  # 2 Hz is not a rate
except ValueError as err:
    print("raised ValueError, as expected:\n    {}".format(err))
else:
    raise AssertionError("2.0 Hz should have raised ValueError")
finally:
    print("bus untouched:", len(bus.log) == n)

A validator refuses a value before any bus traffic.

In [ ]:
def under_100c(v):
    """Refuse limits above 100 C (0x6400 in TMP102 counts)."""
    if v > 0x6400:
        raise ValueError("too hot")


n = len(bus.log)
try:
    dev.reg_set(dev.REG_THIGH, 0x7000, validator=under_100c)  # 112 C
except ValueError as err:
    print("raised ValueError, as expected:\n    {}".format(err))
else:
    raise AssertionError("0x7000 should have raised ValueError")
finally:
    print("bus untouched:", len(bus.log) == n)

#### unit

In [ ]:
%%extract ../tests/test_protocol.py -a --strip-ipytest
%%ipytest


def test_reg_read_assembles_big_endian(proto):
    assert proto.reg_read(proto.REG_CONFIG) == 0x60A0
    assert proto.reg_read(proto.REG_TEMP) == 0x1900
    assert proto.reg_read(proto.REG_TEMP) & 0x000F == 0

In [ ]:
%%extract ../tests/test_protocol.py -a --strip-ipytest
%%ipytest


def test_cache_miss_hit_refresh(proto):
    assert proto.cached(proto.REG_TEMP) is None
    first = proto.reg_get(proto.REG_TEMP)
    proto.conn.bus.nodes[0][2].regs[0] = 0x1A  # the part converts again (26 C)
    assert proto.reg_get(proto.REG_TEMP) == first  # stale by design
    assert proto.reg_get(proto.REG_TEMP, refresh=True) == 0x1A00

In [ ]:
%%extract ../tests/test_protocol.py -a --strip-ipytest
%%ipytest


def test_cache_hit_costs_no_bus_traffic(proto):
    proto.reg_get(proto.REG_CONFIG)
    n = len(proto.conn.bus.log)
    proto.reg_get(proto.REG_CONFIG)
    assert len(proto.conn.bus.log) == n

In [ ]:
%%extract ../tests/test_protocol.py -a --strip-ipytest
%%ipytest


def test_cache_respects_width(proto):
    narrow = proto.reg_get(proto.REG_CONFIG, nbytes=1)
    wide = proto.reg_get(proto.REG_CONFIG, nbytes=2)
    assert (narrow, wide) == (0x60, 0x60A0)

In [ ]:
%%extract ../tests/test_protocol.py -a --strip-ipytest
%%ipytest


def test_cache_clear(proto):
    for reg in (proto.REG_TEMP, proto.REG_CONFIG, proto.REG_THIGH):
        proto.reg_get(reg)
    proto.cache_clear(proto.REG_TEMP)
    assert proto.REG_TEMP not in proto.reg_cache and proto.REG_CONFIG in proto.reg_cache
    proto.cache_clear(0x7F)  # not cached: no error
    proto.cache_clear()
    assert not proto.reg_cache

In [ ]:
%%extract ../tests/test_protocol.py -a --strip-ipytest
%%ipytest


def test_config_verify_mask_excludes_os_res_al(proto):
    excluded = (
        proto.field_mask(proto.F_OS) | proto.field_mask(proto.F_RES) | proto.field_mask(proto.F_AL)
    )
    assert proto.CONFIG_VERIFY_MASK & excluded == 0
    assert 0xFFFF & ~proto.CONFIG_VERIFY_MASK == excluded
    assert proto.verify_mask_for(proto.REG_CONFIG) == proto.CONFIG_VERIFY_MASK
    assert proto.verify_mask_for(proto.REG_THIGH) is None

In [ ]:
%%extract ../tests/test_protocol.py -a --strip-ipytest
%%ipytest


def test_chip_wide_verify_mask_is_the_fallback(proto):
    proto.VERIFY_MASK = 0x0FFF  # set on the instance, so the class stays untouched
    try:
        assert proto.verify_mask_for(proto.REG_THIGH) == 0x0FFF
        assert proto.verify_mask_for(proto.REG_CONFIG) == proto.CONFIG_VERIFY_MASK
    finally:
        del proto.VERIFY_MASK
    assert DemoRegs.VERIFY_MASK is None

In [ ]:
%%extract ../tests/test_protocol.py -a --strip-ipytest
%%ipytest


def test_reg_set_masked_vs_exact(proto):
    original = proto.reg_read(proto.REG_CONFIG)
    sabotaged = original & ~proto.field_mask(proto.F_RES)
    with pytest.raises(ValueError, match="exact|mask 0xffff"):
        proto.reg_set(proto.REG_CONFIG, sabotaged, verify_mask=0xFFFF)
    assert proto.reg_set(proto.REG_CONFIG, sabotaged) == original  # RES reads back 11
    assert proto.cached(proto.REG_CONFIG) == original

In [ ]:
%%extract ../tests/test_protocol.py -a --strip-ipytest
%%ipytest


def test_failed_reg_set_leaves_the_cache_alone(proto):
    proto.reg_get(proto.REG_CONFIG)
    expected = r"reg 0x02: wrote 0xa0, device holds 0x60a0 \(mask 0xffff\)"
    with pytest.raises(ValueError, match=expected):
        proto.reg_set(proto.REG_CONFIG, 0x00A0, verify_mask=0xFFFF)  # RES is read-only
    assert proto.cached(proto.REG_CONFIG) == 0x60A0

In [ ]:
%%extract ../tests/test_protocol.py -a --strip-ipytest
%%ipytest


def test_field_set_leaves_neighbours(proto):
    before = proto.reg_read(proto.REG_CONFIG)
    proto.field_set(proto.REG_CONFIG, proto.F_SD, 1)
    after = proto.reg_read(proto.REG_CONFIG)
    # Only the bits inside the verify mask are the register's settings (OS is status).
    assert (before ^ after) & proto.CONFIG_VERIFY_MASK == proto.field_mask(proto.F_SD)

In [ ]:
%%extract ../tests/test_protocol.py -a --strip-ipytest
%%ipytest


def test_flag_round_trip(proto):
    assert proto.flag_get(proto.REG_CONFIG, proto.F_SD) is False
    proto.flag_set(proto.REG_CONFIG, proto.F_SD, "yes")
    assert proto.flag_get(proto.REG_CONFIG, proto.F_SD) is True
    proto.flag_set(proto.REG_CONFIG, proto.F_SD, 0)
    assert proto.flag_get(proto.REG_CONFIG, proto.F_SD) is False

In [ ]:
%%extract ../tests/test_protocol.py -a --strip-ipytest
%%ipytest


def test_coded_round_trip_and_reject(proto):
    for hz in (0.25, 1.0, 8.0, 4.0):
        proto.coded_set(proto.REG_CONFIG, proto.F_CR, proto.CONV_RATES, hz)
        assert proto.coded_get(proto.REG_CONFIG, proto.F_CR, proto.CONV_RATES) == hz
    with pytest.raises(ValueError, match="not in"):
        proto.coded_set(proto.REG_CONFIG, proto.F_CR, proto.CONV_RATES, 2.0)

In [ ]:
%%extract ../tests/test_protocol.py -a --strip-ipytest
%%ipytest


def test_validator_runs_before_bus_traffic(proto):
    def under_100c(v):
        if v > 0x6400:
            raise ValueError("too hot")

    n = len(proto.conn.bus.log)
    with pytest.raises(ValueError, match="too hot"):
        proto.reg_set(proto.REG_THIGH, 0x7000, validator=under_100c)
    assert len(proto.conn.bus.log) == n
    assert proto.reg_set(proto.REG_THIGH, 0x1E00, validator=under_100c) == 0x1E00

In [ ]:
%%extract ../tests/test_protocol.py -a --strip-ipytest
%%ipytest


def test_tolerance(proto):
    sabotaged = proto.CFG_POR & ~proto.field_mask(proto.F_RES)
    proto.reg_write(proto.REG_CONFIG, sabotaged)
    delta = abs(proto.reg_read(proto.REG_CONFIG) - sabotaged)
    with pytest.raises(ValueError, match="tol"):
        proto.reg_set(proto.REG_CONFIG, sabotaged, tolerance=delta - 1)
    landed = proto.reg_set(proto.REG_CONFIG, sabotaged, tolerance=delta)
    assert proto.cached(proto.REG_CONFIG) == landed

In [ ]:
%%extract ../tests/test_protocol.py -a --strip-ipytest
%%ipytest


def test_reg_write_leaves_cache_stale_and_truncates(proto):
    proto.reg_get(proto.REG_THIGH)
    proto.reg_write(proto.REG_THIGH, 0x2800)
    assert proto.reg_get(proto.REG_THIGH) != 0x2800
    proto.reg_write(proto.REG_THIGH, 0xABCDEF)
    assert proto.reg_read(proto.REG_THIGH) == 0xCDEF

In [ ]:
%%extract ../tests/test_protocol.py -a --strip-ipytest
%%ipytest


def test_endianness_flip_on_instance(proto):
    normal = proto.reg_read(proto.REG_CONFIG)
    proto.BIG_ENDIAN = False
    swapped = proto.reg_read(proto.REG_CONFIG)
    del proto.BIG_ENDIAN
    assert swapped == ((normal & 0xFF) << 8) | (normal >> 8)
    assert DemoRegs.BIG_ENDIAN is True

## `CommandI2CBase`
Command-driven parts: send an opcode (an int, as the datasheet prints it, `CMD_BYTES` wide,
MSB first), then read 16-bit words, each followed by its CRC:

```
[ MSB | LSB | CRC ] [ MSB | LSB | CRC ] ...
```

A command part returns a bare number with nothing to sanity-check it against, so
`words_read` verifies every CRC by default and raises `ValueError` naming the word. The
defaults are Sensirion's (CRC-8, polynomial `0x31`, init `0xFF`); a part from another vendor
overrides the `CRC_*` attributes that differ. Settling delays between a command and its read
belong in the driver.

In [ ]:
%%extract ../src/micropython_i2c/protocol.py -a
class CommandI2CBase(CRCMixin):
    """Command-driven devices: send an opcode, read 16-bit words each followed by a CRC.

    Wire format of a response, checksum per word (so n words is 3n bytes)::

        [ MSB | LSB | CRC ] [ MSB | LSB | CRC ] ...

    A command-driven sensor returns a bare number with nothing to sanity-check it
    against, so ``words_read`` verifies by default and raises on a mismatch.

    Defaults are the Sensirion convention (CRC-8, poly 0x31, init 0xFF). A part from
    another vendor overrides whichever ``CRC_*`` differ.

    Class attributes:
        CMD_BYTES: Opcode width in bytes.
    """

    CMD_BYTES = 2
    CRC_POLY = 0x31
    CRC_INIT = 0xFF

    def cmd_send(self, opcode):
        """Send ``opcode`` (an int, as the datasheet prints it), MSB first.

        Settling delays between command and read are part-specific and belong in the
        driver.
        """
        self.write_raw(self.int_to_bytes(opcode, self.CMD_BYTES, True))

    def words_read(self, n_words, check=True):
        """Read ``n_words`` 16-bit words in one transfer, verifying each word's CRC.

        Args:
            n_words: Number of words expected.
            check: Verify checksums. Turn off only for diagnostics.

        Returns:
            The decoded words as a list of ints.

        Raises:
            ValueError: On the first checksum mismatch, naming the word index.
        """
        raw = self.read_raw(n_words * 3)
        words = []
        for i in range(n_words):
            pair = raw[i * 3 : i * 3 + 2]
            if check:
                self.crc_verify(pair, raw[i * 3 + 2], "word {}".format(i))
            words.append((pair[0] << 8) | pair[1])
        return words

### validation/test
A demo command part: `CommandI2CBase` on an `I2CDevice`, talking to a `FakeSensirion` that
answers two words, each with the Sensirion CRC.

In [ ]:
%%extract ../tests/test_protocol.py -a
class FakeGasSensor(CommandI2CBase, I2CDevice):
    ALLOWED_ADDRS = (0x62,)


def gas_sensor():
    """A ``FakeGasSensor`` on a bus with a ``FakeSensirion`` answering 0xBEEF, 0x1234."""
    bus = FakeBus()
    part = bus.attach(0x62, FakeSensirion([0xBEEF, 0x1234], CommandI2CBase.crc))
    return FakeGasSensor(I2CConnection(bus, 0x62)), part

#### validation: nominal usage
Send a command, read both words; the bus log shows one write of two bytes and one read of six.

In [ ]:
gas, part = gas_sensor()
gas.cmd_send(0x2C06)
print("commands seen:", [hex(c) for c in part.commands], "| words:", [hex(w) for w in gas.words_read(2)])
print("bus log:", gas.conn.bus.log)
print("CRC of 0xBEEF (datasheet example 0x92):", hex(CommandI2CBase.crc(b"\xbe\xef")))

#### validation: anti-nominal usage
A corrupted checksum on the first word is refused, naming the word.

In [ ]:
part.crc_fn = lambda pair: CommandI2CBase.crc(pair) ^ 1
try:
    gas.words_read(2)  # every CRC is off by one bit
except ValueError as err:
    print("raised ValueError, as expected:\n    {}".format(err))
else:
    raise AssertionError("a bad CRC should have raised ValueError")
finally:
    print("unchecked read still decodes:", [hex(w) for w in gas.words_read(2, check=False)])

#### unit

In [ ]:
%%extract ../tests/test_protocol.py -a --strip-ipytest
%%ipytest


def test_words_read_verifies_each_word():
    bus = FakeBus()
    part = bus.attach(0x62, FakeSensirion([0xBEEF, 0x1234], CommandI2CBase.crc))
    dev = FakeGasSensor(I2CConnection(bus, 0x62))
    dev.cmd_send(0x2C06)
    assert part.commands == [0x2C06]
    assert dev.words_read(2) == [0xBEEF, 0x1234]

    part.crc_fn = lambda pair: CommandI2CBase.crc(pair) ^ 1
    with pytest.raises(ValueError, match="word 0"):
        dev.words_read(2)
    assert dev.words_read(2, check=False) == [0xBEEF, 0x1234]

In [ ]:
%%extract ../tests/test_protocol.py -a --strip-ipytest
%%ipytest


def test_bad_crc_on_a_later_word_names_that_word():
    dev, part = gas_sensor()
    part.crc_fn = lambda pair: CommandI2CBase.crc(pair) ^ (pair == b"\x12\x34")
    assert dev.words_read(1) == [0xBEEF]
    with pytest.raises(ValueError, match="word 1"):
        dev.words_read(2)

In [ ]:
%%extract ../tests/test_protocol.py -a --strip-ipytest
%%ipytest


def test_words_read_is_one_transfer_of_three_bytes_per_word():
    dev, _ = gas_sensor()
    dev.words_read(2)
    assert dev.conn.bus.log == [("r", 0x62, 6)]

In [ ]:
%%extract ../tests/test_protocol.py -a --strip-ipytest
%%ipytest


def test_cmd_send_width_follows_cmd_bytes():
    dev, part = gas_sensor()
    dev.cmd_send(0x2C06)
    dev.CMD_BYTES = 1  # an 8-bit opcode part
    dev.cmd_send(0x1F)
    assert dev.conn.bus.log == [("w", 0x62, b"\x2c\x06"), ("w", 0x62, b"\x1f")]
    assert part.commands == [0x2C06, 0x1F]

In [ ]:
%%extract ../tests/test_protocol.py -a --strip-ipytest
%%ipytest


def test_sensirion_crc_defaults():
    assert (CommandI2CBase.CRC_POLY, CommandI2CBase.CRC_INIT) == (0x31, 0xFF)
    assert CommandI2CBase.crc(b"\xbe\xef") == 0x92  # Sensirion datasheet example

# Hardware validation

> ## ⚠️ STOP — do not run this section unattended
>
> **Everything below talks to the real board.** It connects to the FeatherS3, uploads the
> package, and on I2C bus 1 selects and releases channel 2 of the TCA9548A at `0x70` and reads
> the TMP102 at `0x48` (temperature, configuration and T-high registers). Nothing is written
> to the TMP102: the one `reg_set` below is refused by its validator before any bus traffic,
> and would write back the value just read if it were not.
>
> - Run it **one cell at a time**. Read the **What to do** line above each cell first.
> - The section ends with `%board disconnect`. If you stop partway, run that cell yourself.

**Hardware and configuration** (same as the table at the top):

| | Host run (simulation) | Hardware run |
|---|---|---|
| **Runs on** | CPython 3.13 in the micropylab kernel, no board | Unexpected Maker FeatherS3 (ESP32-S3), MicroPython 1.28.0, USB `/dev/ttyACM0` |
| **I2C bus** | `FakeBus` | `machine.I2C(1)`, **bus 1**, scl=15 sda=16, 100 kHz (`bus_open("feathers3", 1)`) |
| **Mux** | none | SparkFun Qwiic Mux (TCA9548A) at `0x70`, channel 2 |
| **Register part** | `FakeRegisterPart` at `0x48` laid out like a TMP102 (16-bit registers at `0x00`, `0x02`, `0x04`; configuration `0x60A0` with read-only resolution and status bits) | TMP102 at `0x48`: temperature and configuration registers read, never written |
| **Command part** | `FakeSensirion` at `0x62`, words `0xBEEF`, `0x1234`, Sensirion CRC-8 | a stand-in bus on the board (no command part on bus 1) |
| **Result** | not run yet | not run yet |

Layout: connect and set up, the **matching tests** (one per unit test above that needs no
write, same name), then the tally and disconnect. Tests that write a register
(`reg_set`, `field_set`, `flag_set`, `coded_set`, `reg_write`) have no match: this section is
read-only.

## Connect and set up

Notebook-only: connect to the board. Only one notebook can own the board: if another notebook
still has it, run `%board disconnect` there first.

**What to do:** make sure the board is plugged in and no other notebook owns it.

In [ ]:
%board connect

Notebook-only: remove the package an earlier upload left on the board, and make sure `/lib`
exists, so the copy below lands exactly at `/lib/micropython_i2c`.

**What to do:** nothing.

In [ ]:
import os


def rmtree(path):
    """Delete a board folder and everything in it."""
    for entry in os.ilistdir(path):
        child = path + "/" + entry[0]
        if entry[1] == 0x4000:  # folder
            rmtree(child)
        else:
            os.remove(child)
    os.rmdir(path)


try:
    rmtree("/lib/micropython_i2c")
    print("removed the old /lib/micropython_i2c")
except OSError:
    print("no /lib/micropython_i2c on the board yet")
try:
    os.mkdir("/lib")
except OSError:
    pass  # already there

Notebook-only: copy `src/micropython_i2c`, including the file this notebook just wrote, to
`/lib/micropython_i2c`. The copy runs on the host; the path is relative to this notebook's
folder.

**What to do:** nothing. Takes a few seconds.

In [ ]:
%%host
from pathlib import Path

pkg = next(p for p in (Path("../src"), Path("src")) if p.is_dir()) / "micropython_i2c"
%files copy -r {pkg} :/lib/micropython_i2c

Notebook-only: drop any modules an earlier run left on the board, so the new files load, then
import what the checks use and zero the tally.

**What to do:** nothing.

In [ ]:
import sys

for name in [m for m in sys.modules if m.startswith("micropython_i2c")]:
    del sys.modules[name]

from micropython_i2c.boards import bus_open
from micropython_i2c.compat import default_lock
from micropython_i2c.connection import I2CConnection, path_release
from micropython_i2c.debug import check, check_raises, check_reset, check_summary, info
from micropython_i2c.debug import check_counts  # noqa: F401  (read from the host with %eval)
from micropython_i2c.node import I2CDevice
from micropython_i2c.protocol import CommandI2CBase, RegisterI2CBase

check_reset()

Notebook-only: the bus, the bench constants and the demo classes. `BenchRegs` is the TMP102's
real register map on `RegisterI2CBase` (not the TMP102 driver, so only this module is under
test). `CountingBus` counts the transfers that reach the bus, for the cache and validator
checks. The mux starts deselected. If the bench is rewired, change the constants.

**What to do:** nothing.

In [ ]:
BUS_ID, MUX_ADDR, TMP_ADDR = 1, 0x70, 0x48
TMP_PATH = [[MUX_ADDR, 2]]

bus = bus_open("feathers3", BUS_ID, freq=100_000)
bus_lock = default_lock()
path_release(bus, TMP_PATH)


class CountingBus:
    """The bus, counting every transfer in ``n``."""

    def __init__(self, inner):
        self.inner = inner
        self.n = 0

    def writeto(self, addr, data, *args):
        self.n += 1
        return self.inner.writeto(addr, data, *args)

    def readfrom_mem(self, addr, reg, nbytes, addrsize=8):
        self.n += 1
        return self.inner.readfrom_mem(addr, reg, nbytes, addrsize=addrsize)

    def writeto_mem(self, addr, reg, data, addrsize=8):
        self.n += 1
        self.inner.writeto_mem(addr, reg, data, addrsize=addrsize)


class BenchRegs(RegisterI2CBase, I2CDevice):
    """The TMP102's register map (datasheet SBOS397), read-only use."""

    ALLOWED_ADDRS = (0x48, 0x49, 0x4A, 0x4B)
    REG_TEMP, REG_CONFIG, REG_TLOW, REG_THIGH = 0x00, 0x01, 0x02, 0x03
    F_OS = (15, 1)
    F_RES = (13, 2)  # read-only, always 0b11
    F_CR = (6, 2)
    F_AL = (5, 1)
    F_EM = (4, 1)
    CONV_RATES = {0: 0.25, 1: 1.0, 2: 4.0, 3: 8.0}
    CONFIG_VERIFY_MASK = 0xFFFF & ~((1 << 15) | (3 << 13) | (1 << 5))
    VERIFY_MASKS = {REG_CONFIG: CONFIG_VERIFY_MASK}


counting = CountingBus(bus)
tmp = BenchRegs(I2CConnection(counting, TMP_ADDR, TMP_PATH, bus_lock))
print(tmp)

## Matching tests

**Matches** `test_reg_read_assembles_big_endian`: the resolution bits read `11`, and in
normal (12-bit) mode the temperature's low four bits are 0. The value is printed as a room
reading.

**What to do:** nothing.

In [ ]:
cfg = tmp.reg_read(tmp.REG_CONFIG)
temp = tmp.reg_read(tmp.REG_TEMP)
check("RES bits read 11", tmp.field_extract(cfg, tmp.F_RES) == 3, "config {:#06x}".format(cfg))
if tmp.field_extract(cfg, tmp.F_EM):
    info("extended mode: low bits carry the 13th bit, skipped")
else:
    check("test_reg_read_assembles_big_endian", temp & 0x000F == 0,
          "{:#06x} = {:.2f} C".format(temp, tmp.sign_extend(temp >> 4, 12) * 0.0625))  # fmt: skip

**Matches** `test_cache_miss_hit_refresh` and `test_cache_hit_costs_no_bus_traffic`.

**What to do:** nothing.

In [ ]:
tmp.cache_clear()
check("miss: nothing cached", tmp.cached(tmp.REG_CONFIG) is None)
first = tmp.reg_get(tmp.REG_CONFIG)
n = counting.n
check("hit: same value", tmp.reg_get(tmp.REG_CONFIG) == first)
check("test_cache_hit_costs_no_bus_traffic", counting.n == n)
check("refresh: reads the bus", tmp.reg_get(tmp.REG_CONFIG, refresh=True) == cfg and counting.n > n)

**Matches** `test_cache_respects_width`: a 1-byte read of the configuration register is its
high byte, and is not served from the 2-byte cache entry.

**What to do:** nothing.

In [ ]:
wide = tmp.reg_get(tmp.REG_CONFIG, nbytes=2)
narrow = tmp.reg_get(tmp.REG_CONFIG, nbytes=1)
check("test_cache_respects_width", narrow == wide >> 8, "{:#04x} / {:#06x}".format(narrow, wide))

**Matches** `test_cache_clear`.

**What to do:** nothing.

In [ ]:
for reg in (tmp.REG_TEMP, tmp.REG_CONFIG, tmp.REG_THIGH):
    tmp.reg_get(reg)
tmp.cache_clear(tmp.REG_TEMP)
ok = tmp.REG_TEMP not in tmp.reg_cache and tmp.REG_CONFIG in tmp.reg_cache
tmp.cache_clear(0x7F)
tmp.cache_clear()
check("test_cache_clear", ok and not tmp.reg_cache)

**Matches** `test_config_verify_mask_excludes_os_res_al`.

**What to do:** nothing.

In [ ]:
excluded = tmp.field_mask(tmp.F_OS) | tmp.field_mask(tmp.F_RES) | tmp.field_mask(tmp.F_AL)
check("test_config_verify_mask_excludes_os_res_al",
      tmp.CONFIG_VERIFY_MASK & excluded == 0 and 0xFFFF & ~tmp.CONFIG_VERIFY_MASK == excluded
      and tmp.verify_mask_for(tmp.REG_CONFIG) == tmp.CONFIG_VERIFY_MASK
      and tmp.verify_mask_for(tmp.REG_THIGH) is None)  # fmt: skip

**Matches** `test_validator_runs_before_bus_traffic`, refusing half only: the validator
refuses every value, so nothing is written. The value is the one T-high already holds, so
even a broken refusal would write back the same value.

**What to do:** nothing.

In [ ]:
def refuse(v):
    raise ValueError("refused {:#06x}".format(v))


thigh = tmp.reg_read(tmp.REG_THIGH)
n = counting.n
check_raises("validator refuses", ValueError, tmp.reg_set, tmp.REG_THIGH, thigh, validator=refuse)
check("test_validator_runs_before_bus_traffic", counting.n == n)

**Matches** `test_endianness_flip_on_instance` and the read halves of `test_flag_round_trip`
and `test_coded_round_trip_and_reject`: the conversion rate maps through the table, the
alert flag reads as a bool.

**What to do:** nothing.

In [ ]:
normal = tmp.reg_read(tmp.REG_CONFIG)
tmp.BIG_ENDIAN = False
swapped = tmp.reg_read(tmp.REG_CONFIG)
del tmp.BIG_ENDIAN
check("test_endianness_flip_on_instance",
      swapped == ((normal & 0xFF) << 8) | (normal >> 8) and BenchRegs.BIG_ENDIAN is True)  # fmt: skip
rate = tmp.coded_get(tmp.REG_CONFIG, tmp.F_CR, tmp.CONV_RATES)
check("coded_get maps through the table", rate in tmp.CONV_RATES.values(), "{} Hz".format(rate))
check("flag_get returns a bool", tmp.flag_get(tmp.REG_CONFIG, tmp.F_AL) in (True, False))
check_raises("coded_set refuses 2.0 Hz", ValueError,
             tmp.coded_set, tmp.REG_CONFIG, tmp.F_CR, tmp.CONV_RATES, 2.0)  # fmt: skip

**Matches** `test_words_read_verifies_each_word`, `test_bad_crc_on_a_later_word_names_that_word`,
`test_words_read_is_one_transfer_of_three_bytes_per_word`, `test_cmd_send_width_follows_cmd_bytes`
and `test_sensirion_crc_defaults`. There is no command part on bus 1, so a stand-in bus on the
board answers like a Sensirion part: this runs the CRC and word decoding on MicroPython.

**What to do:** nothing.

In [ ]:
class WordsBus:
    """Answers reads with ``words``, each followed by ``crc_fn(pair)``; logs every transfer."""

    def __init__(self, words, crc_fn):
        self.words = words
        self.crc_fn = crc_fn
        self.log = []

    def writeto(self, addr, data, *args):
        self.log.append(("w", addr, bytes(data)))
        return len(data)

    def readfrom(self, addr, nbytes, *args):
        self.log.append(("r", addr, nbytes))
        out = b""
        for w in self.words:
            pair = bytes([w >> 8, w & 0xFF])
            out += pair + bytes([self.crc_fn(pair)])
        return out[:nbytes]


class BenchGas(CommandI2CBase, I2CDevice):
    ALLOWED_ADDRS = (0x62,)


wbus = WordsBus([0xBEEF, 0x1234], CommandI2CBase.crc)
gas = BenchGas(I2CConnection(wbus, 0x62))
gas.cmd_send(0x2C06)
check("test_sensirion_crc_defaults", CommandI2CBase.crc(b"\xbe\xef") == 0x92)
check("cmd_send writes the opcode MSB first", wbus.log == [("w", 0x62, b"\x2c\x06")])
check("test_words_read_verifies_each_word", gas.words_read(2) == [0xBEEF, 0x1234])
check("one read of 6 bytes", wbus.log[-1] == ("r", 0x62, 6))
wbus.crc_fn = lambda pair: CommandI2CBase.crc(pair) ^ 1
check_raises("bad CRC on word 0 refused", ValueError, gas.words_read, 2)
check("check=False still decodes", gas.words_read(2, check=False) == [0xBEEF, 0x1234])
wbus.crc_fn = lambda pair: CommandI2CBase.crc(pair) ^ (1 if pair == b"\x12\x34" else 0)
err = check_raises("bad CRC on word 1 refused", ValueError, gas.words_read, 2)
check("test_bad_crc_on_a_later_word_names_that_word", err is not None and "word 1" in str(err))

## Tally and disconnect

Notebook-only: leave the mux deselected, then the PASS/FAIL tally of every check in this
section.

**What to do:** read it. Any `FAIL` line above says which check and why.

In [ ]:
path_release(bus, TMP_PATH)
check_summary()

Notebook-only: copy the tally to the host before disconnecting, so the last cell can fail the
run after the board is released.

**What to do:** nothing.

In [ ]:
%%host
hw_passed, hw_failed = get_ipython().run_line_magic("eval", "check_counts()")

Notebook-only: release the board, so the next run (or a kernel restart) starts on the host.

**What to do:** nothing.

In [ ]:
%board disconnect

Notebook-only: fail the run if any hardware check failed. It runs on the host.

**What to do:** nothing. An error here means a `FAIL` line above.

In [ ]:
%%host
if hw_failed:
    raise AssertionError("{} of {} hardware checks failed".format(hw_failed, hw_passed + hw_failed))
print("all {} hardware checks passed".format(hw_passed))